In [ ]:
!wget -q https://github.com/ShivamShrirao/diffusers/raw/main/examples/dreambooth/train_dreambooth.py
!wget -q https://github.com/ShivamShrirao/diffusers/raw/main/scripts/convert_diffusers_to_original_stable_diffusion.py
%pip install -qq git+https://github.com/ShivamShrirao/diffusers
%pip install -q -U --pre triton
%pip install -q accelerate transformers ftfy bitsandbytes natsort safetensors xformers

In [ ]:
import json
import os

from google.colab import files
import shutil

from natsort import natsorted
from glob import glob

import torch
from torch import autocast
from diffusers import StableDiffusionPipeline, DDIMScheduler
from IPython.display import display

In [ ]:
!mkdir -p ~/.huggingface
HUGGINGFACE_TOKEN = "put your token here"
!echo -n "{HUGGINGFACE_TOKEN}" > ~/.huggingface/token

In [ ]:
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "/content/stable_diffusion_weights/zwx"

# Create output directory
!mkdir -p $OUTPUT_DIR

In [ ]:
# The concepts_list is a list of concepts/subject, each represented as a dictionary
concepts_list = [
    {
        "instance_prompt":   "photo of zwx dog",
        "class_prompt":      "photo of a dog",
        "instance_data_dir": "/content/data/zwx",
        "class_data_dir":    "/content/data/dog"
    },
]

# Create a directory for each concept according to its instance_data_dir
for c in concepts_list:
    os.makedirs(c["instance_data_dir"], exist_ok=True)

# Dump the concepts_list to a JSON file
with open("concepts_list.json", "w") as f:
    json.dump(concepts_list, f, indent=4)

In [ ]:
!python3 train_dreambooth.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --pretrained_vae_name_or_path="stabilityai/sd-vae-ft-mse" \
  --output_dir=$OUTPUT_DIR \
  --revision="fp16" \
  --with_prior_preservation --prior_loss_weight=1.0 \
  --seed=1337 \
  --resolution=512 \
  --train_batch_size=1 \
  --train_text_encoder \
  --mixed_precision="fp16" \
  --use_8bit_adam \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-6 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --num_class_images=50 \
  --sample_batch_size=4 \
  --max_train_steps=800 \
  --save_interval=10000 \
  --save_sample_prompt="photo of zwx dog" \
  --concepts_list="concepts_list.json"

In [ ]:
WEIGHTS_DIR = natsorted(glob(OUTPUT_DIR + os.sep + "*"))[-1]
ckpt_path = WEIGHTS_DIR + "/model.ckpt"

half_arg = ""
fp16 = True
if fp16:
    half_arg = "--half"
!python convert_diffusers_to_original_stable_diffusion.py \
    --model_path $WEIGHTS_DIR  --checkpoint_path $ckpt_path $half_arg
print(f"[*] Converted ckpt saved at {ckpt_path}")

In [ ]:
model_path = WEIGHTS_DIR

pipe = StableDiffusionPipeline.from_pretrained(model_path, safety_checker=None,
                                               torch_dtype=torch.float16
                                              ).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()

g_cuda = torch.Generator(device='cuda')
seed = 52362
g_cuda.manual_seed(seed)

In [ ]:
prompt = "photo of zwx dog in a bucket wearing 3d glasses"
negative_prompt = ""
num_samples = 4
guidance_scale = 7.5
num_inference_steps = 24
height = 512
width = 512

with autocast("cuda"), torch.inference_mode():
    images = pipe(
        prompt,
        height=height,
        width=width,
        negative_prompt=negative_prompt,
        num_images_per_prompt=num_samples,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=g_cuda
    ).images

for img in images:
    display(img)

# Free runtime memory
exit()